In [1]:
from pathlib import Path

print(Path.cwd())
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

DATA_DIR.mkdir(exist_ok=True)

print("Dataset location:", DATA_DIR)

C:\Users\user\deep-learning-traffic-signs\notebooks
Dataset location: C:\Users\user\deep-learning-traffic-signs\data


In [2]:
from torchvision import transforms 
basic_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [3]:
from torchvision.datasets import GTSRB

train_dataset = GTSRB(
    root=str(DATA_DIR),
    split="train",
    transform=basic_transform,
    download=True
)

test_dataset = GTSRB(
    root=str(DATA_DIR),
    split="test",
    transform=basic_transform,
    download=True
)


In [4]:
from torch.utils.data import DataLoader

In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [6]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
 
)

In [7]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([32, 3, 32, 32])
Labels shape: torch.Size([32])


In [8]:
batch=next(iter(train_loader))
batch

[tensor([[[[0.1686, 0.1686, 0.1686,  ..., 0.1922, 0.2235, 0.2039],
           [0.1608, 0.1529, 0.1529,  ..., 0.1725, 0.2196, 0.1804],
           [0.1451, 0.1412, 0.1490,  ..., 0.1490, 0.1804, 0.1608],
           ...,
           [0.1255, 0.1294, 0.1216,  ..., 0.1451, 0.1412, 0.1255],
           [0.1294, 0.1255, 0.1255,  ..., 0.1529, 0.1490, 0.1294],
           [0.1294, 0.1333, 0.1294,  ..., 0.1608, 0.1490, 0.1333]],
 
          [[0.1412, 0.1451, 0.1451,  ..., 0.1765, 0.1961, 0.1804],
           [0.1373, 0.1333, 0.1333,  ..., 0.1686, 0.2078, 0.1608],
           [0.1255, 0.1216, 0.1294,  ..., 0.1529, 0.1765, 0.1451],
           ...,
           [0.1059, 0.1176, 0.1176,  ..., 0.1255, 0.1176, 0.1098],
           [0.1137, 0.1176, 0.1216,  ..., 0.1412, 0.1333, 0.1176],
           [0.1176, 0.1255, 0.1255,  ..., 0.1529, 0.1333, 0.1255]],
 
          [[0.1294, 0.1333, 0.1333,  ..., 0.2118, 0.1961, 0.1490],
           [0.1255, 0.1255, 0.1255,  ..., 0.2549, 0.2314, 0.1216],
           [0.1176, 0.11

In [9]:
import torch

In [10]:
device=torch.device("cuda"if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [11]:
import torch
import torch.nn as nn
from torch.optim import Adam
import  torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F 
import matplotlib.pyplot as plt


In [12]:
class MyConvBlock( nn.Module ):
    def __init__(self,in_ch,out_ch,dropout_p):
        kernel_size=3
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size,stride=1,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2,stride=2)
        )
    def forward(self,x):
        return self.model(x)
        

In [13]:
flattened_img_size=128*4*4
N_CLASSES=43
IMG_CHS=3
IMG_WIDH=32
IMG_LENGHT=32
base_model= nn.Sequential(
    MyConvBlock(IMG_CHS,32,0), #(32,16,16)
    MyConvBlock(32,64,0.2),#(64,8,8)
    MyConvBlock(64,128,0),#(128,4,4)
    nn.Flatten(),
    nn.Linear(flattened_img_size,512),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(512,N_CLASSES)
)
    

In [14]:
loss_function=nn.CrossEntropyLoss()
optimizer=Adam(base_model.parameters())

In [15]:
train_N=len(train_loader)
test_N=len(test_loader)

In [16]:

import torch
torch._dynamo.config.suppress_errors = True
model=torch.compile(base_model.to(device))
model

OptimizedModule(
  (_orig_mod): Sequential(
    (0): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.2, inplace=False)
        (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): MyConvBlock(
      (model): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(128, eps=1e

In [17]:
def get_batch_accuracy(output,y,N):
    pred=output.argmax(dim=1,keepdim=True)
    correct=pred.eq(y.view_as(pred)).sum().item()
    return correct/N


In [18]:
def train():
    loss=0
    accuracy=0
    model.train()
    for x,y in train_loader:
        output=model(x)
        optimizer.zero_grad()
        batch_loss=loss_function(output,y)
        batch_loss.backward()
        optimizer.step()
        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output,y,train_N)
    print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))

In [19]:
def validate():
    loss=0
    accuracy=0
    model.eval()
    with torch.no_grad():
        for x,y in test_loader:
            output=model(x)
            loss+=loss_function(output,y).item()
            accuracy += get_batch_accuracy(output,y,test_N)
        print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))

In [20]:
epochs=20
for epoch in range (epochs):
    print ('epoch:{}'.format(epoch))   
    train()
    validate()

epoch:0


W0920 05:40:45.201000 10336 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] WON'T CONVERT inner C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\external_utils.py line 67 
W0920 05:40:45.201000 10336 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] due to: 
W0920 05:40:45.201000 10336 Lib\site-packages\torch\_dynamo\convert_frame.py:2429] Traceback (most recent call last):
W0920 05:40:45.201000 10336 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]   File "C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torch\_dynamo\convert_frame.py", line 2333, in __call__
W0920 05:40:45.201000 10336 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     result = self._inner_convert(
W0920 05:40:45.201000 10336 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]         frame, cache_entry, hooks, frame_state, skip=skip + 1
W0920 05:40:45.201000 10336 Lib\site-packages\torch\_dynamo\convert_frame.py:2429]     )
W0920 05:40:45.2

Valid-Loss:874.7934 Accuracy 21.8439
Valid-Loss:158.2902 Accuracy 28.2987
epoch:1
Valid-Loss:131.5076 Accuracy 30.3745
Valid-Loss:167.8163 Accuracy 28.4177
epoch:2
Valid-Loss:65.2978 Accuracy 31.1777
Valid-Loss:78.2992 Accuracy 30.1671
epoch:3
Valid-Loss:57.8625 Accuracy 31.2713
Valid-Loss:91.5541 Accuracy 29.9291
epoch:4
Valid-Loss:42.1723 Accuracy 31.4922
Valid-Loss:75.2742 Accuracy 30.4405
epoch:5
Valid-Loss:37.0541 Accuracy 31.5342
Valid-Loss:79.6251 Accuracy 30.4911
epoch:6
Valid-Loss:25.4929 Accuracy 31.6687
Valid-Loss:99.8579 Accuracy 29.9899
epoch:7
Valid-Loss:32.2111 Accuracy 31.6146
Valid-Loss:92.7798 Accuracy 30.1772
epoch:8
Valid-Loss:17.5496 Accuracy 31.7683
Valid-Loss:81.9593 Accuracy 30.2354
epoch:9
Valid-Loss:23.1548 Accuracy 31.7035
Valid-Loss:81.8564 Accuracy 30.3772
epoch:10
Valid-Loss:17.8284 Accuracy 31.7575
Valid-Loss:90.8023 Accuracy 30.3873
epoch:11
Valid-Loss:17.5733 Accuracy 31.7815
Valid-Loss:85.4276 Accuracy 30.5519
epoch:12
Valid-Loss:17.6659 Accuracy 31.76